In [ ]:

import sys
import math

from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

from src.streaming.event_contract import (
    validate_event,
    serialize_event,
    deserialize_event,
    get_event_key,
    EventValidationError
)


# ============================================================
# 1. EVENTO DE REFERENCIA
# ============================================================

valid_event = {
    "event_id": "HAPPINESS-2019-COLOMBIA",
    "country": "Colombia",
    "year": 2019,
    "gdp": 1.002,
    "social_support": 1.237,
    "health": 0.795,
    "freedom": 0.535,
    "corruption": 0.091,
    "generosity": 0.099,
    "actual_happiness_score": 6.125,
    "event_time": "2026-11-06T21:30:00Z"
}


# ============================================================
# 2. CASOS DE VALIDACIÓN
# ============================================================

test_cases = []


def add_test(name, event, expected_status):
    result = validate_event(event)

    test_cases.append({
        "test": name,
        "expected": expected_status,
        "obtained": result["status"],
        "passed": result["status"] == expected_status,
        "errors": result["errors"]
    })


# Evento válido
add_test(
    "Evento completo",
    valid_event.copy(),
    "VALID"
)


# corruption admite null
event = valid_event.copy()
event["corruption"] = None

add_test(
    "Corruption nulo",
    event,
    "VALID"
)


# Campo obligatorio ausente
event = valid_event.copy()
del event["social_support"]

add_test(
    "Campo ausente",
    event,
    "INVALID_SCHEMA"
)


# Campo adicional
event = valid_event.copy()
event["region"] = "South America"

add_test(
    "Campo adicional",
    event,
    "INVALID_SCHEMA"
)


# Null en un predictor no nullable
event = valid_event.copy()
event["social_support"] = None

add_test(
    "Null en social_support",
    event,
    "INVALID_SCHEMA"
)


# Indicador negativo
event = valid_event.copy()
event["gdp"] = -0.5

add_test(
    "Indicador negativo",
    event,
    "INVALID_VALUES"
)


# Score superior a 10
event = valid_event.copy()
event["actual_happiness_score"] = 11.5

add_test(
    "Score fuera de rango",
    event,
    "INVALID_VALUES"
)


# Año fuera del periodo
event = valid_event.copy()
event["year"] = 2020

add_test(
    "Año fuera del periodo",
    event,
    "INVALID_VALUES"
)


# Timestamp sin UTC Z
event = valid_event.copy()
event["event_time"] = "2026-11-06T21:30:00"

add_test(
    "Timestamp sin Z",
    event,
    "INVALID_VALUES"
)


# NaN
event = valid_event.copy()
event["gdp"] = math.nan

add_test(
    "Valor NaN",
    event,
    "INVALID_VALUES"
)


# ============================================================
# 3. PRESENTACIÓN DE RESULTADOS
# ============================================================

print("=" * 75)
print("VALIDACIÓN DEL CONTRATO DE EVENTOS")
print("=" * 75)

for test in test_cases:

    status = "OK" if test["passed"] else "ERROR"

    print(
        f"[{status}] {test['test']:<30} "
        f"Esperado: {test['expected']:<15} "
        f"Obtenido: {test['obtained']}"
    )

print("=" * 75)

total = len(test_cases)
passed = sum(test["passed"] for test in test_cases)

print(f"Pruebas ejecutadas: {total}")
print(f"Pruebas correctas:  {passed}")
print(f"Pruebas fallidas:   {total - passed}")


# ============================================================
# 4. PRUEBA DE SERIALIZACIÓN
# ============================================================

print("\n" + "=" * 75)
print("PRUEBA DE SERIALIZACIÓN Y DESERIALIZACIÓN")
print("=" * 75)

serialized = serialize_event(valid_event)

deserialized = deserialize_event(serialized)

print("Tipo serializado:", type(serialized).__name__)
print("Codificación:", "UTF-8")
print("Evento recuperado:", deserialized == valid_event)


# ============================================================
# 5. PRUEBA DE CLAVE KAFKA
# ============================================================

print("\n" + "=" * 75)
print("PRUEBA DE CLAVE KAFKA")
print("=" * 75)

event_key = get_event_key(valid_event)

print("Clave:", event_key)
print("Tipo:", type(event_key).__name__)
print("Valor decodificado:", event_key.decode("utf-8"))


# ============================================================
# 6. PRUEBA DE JSON MAL FORMADO
# ============================================================

print("\n" + "=" * 75)
print("PRUEBA DE JSON INVÁLIDO")
print("=" * 75)

try:

    deserialize_event(b'{"country": "Colombia"')

except EventValidationError as error:

    print("Estado:", error.status)
    print("Error:", error.errors[0]["message"])


ModuleNotFoundError: No module named 'streaming'